# 🚀 Train DeBERTa v3 for Resume NER

This notebook trains a DeBERTa v3 model on your Label Studio annotations for resume parsing.

**Labels:**
- Work Experience: COMPANY, ROLE, DATE_START, DATE_END, LOCATION, CLIENT
- Education: DEGREE, INSTITUTION, FIELD, GRADE, EDU_YEAR_START, EDU_YEAR_END

**Steps:**
1. Upload your Label Studio JSON files
2. Convert to Hugging Face format
3. Train DeBERTa v3 model
4. Download trained model

## 1️⃣ Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate evaluate seqeval

## 2️⃣ Upload Label Studio JSON Files

**Upload your Label Studio export files here:**
- Click the folder icon on the left
- Upload your `project-*.json` files from Label Studio
- Or upload a folder containing all JSON files

In [ ]:
from google.colab import files
import json
import os

# Upload Label Studio JSON files
print("📤 Upload your Label Studio JSON files...")
uploaded = files.upload()

print(f"\n✅ Uploaded {len(uploaded)} files")
for filename in uploaded.keys():
    print(f"  - {filename}")

## 3️⃣ Convert Label Studio to Hugging Face Format

In [ ]:
import json
from collections import defaultdict

# Define label mappings
LABELS = [
    "O",
    "B-COMPANY", "I-COMPANY",
    "B-CLIENT", "I-CLIENT",
    "B-ROLE", "I-ROLE",
    "B-LOCATION", "I-LOCATION",
    "B-DATE_START", "I-DATE_START",
    "B-DATE_END", "I-DATE_END",
    "B-DEGREE", "I-DEGREE",
    "B-FIELD", "I-FIELD",
    "B-INSTITUTION", "I-INSTITUTION",
    "B-EDU_YEAR_START", "I-EDU_YEAR_START",
    "B-EDU_YEAR_END", "I-EDU_YEAR_END",
    "B-GRADE", "I-GRADE"
]

label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for idx, label in enumerate(LABELS)}

print(f"📊 Total labels: {len(LABELS)}")
print(f"Labels: {LABELS[:10]}...")

In [ ]:
def convert_label_studio_to_ner(json_files):
    """
    Convert Label Studio JSON export to NER format.
    
    Label Studio format:
    {
      "data": {"text": "..."},
      "annotations": [{
        "result": [{
          "value": {"start": 0, "end": 10, "text": "Google", "labels": ["COMPANY"]}
        }]
      }]
    }
    """
    examples = []
    
    for json_file in json_files:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Handle both single object and array formats
        if isinstance(data, dict):
            data = [data]
        
        for item in data:
            text = item.get('data', {}).get('text', '')
            if not text:
                continue
            
            # Get annotations
            annotations = item.get('annotations', [])
            if not annotations:
                continue
            
            # Use first annotation (or merge if multiple)
            entities = []
            for ann in annotations:
                for result in ann.get('result', []):
                    value = result.get('value', {})
                    if 'start' in value and 'end' in value and 'labels' in value:
                        entities.append({
                            'start': value['start'],
                            'end': value['end'],
                            'label': value['labels'][0]  # Take first label
                        })
            
            if entities:
                examples.append({
                    'text': text,
                    'entities': sorted(entities, key=lambda x: x['start'])
                })
    
    return examples

# Convert all uploaded JSON files
json_files = [f for f in uploaded.keys() if f.endswith('.json')]
examples = convert_label_studio_to_ner(json_files)

print(f"\n✅ Converted {len(examples)} annotated examples")
print(f"\nSample example:")
if examples:
    sample = examples[0]
    print(f"Text: {sample['text'][:100]}...")
    print(f"Entities: {sample['entities'][:3]}")

In [ ]:
def create_ner_tags(text, entities):
    """
    Convert character-level entities to token-level BIO tags.
    """
    # Simple whitespace tokenization
    tokens = text.split()
    tags = ['O'] * len(tokens)
    
    # Track character positions
    char_to_token = {}
    char_pos = 0
    for token_idx, token in enumerate(tokens):
        for i in range(len(token)):
            char_to_token[char_pos + i] = token_idx
        char_pos += len(token) + 1  # +1 for space
    
    # Assign BIO tags
    for entity in entities:
        start_token = char_to_token.get(entity['start'])
        end_token = char_to_token.get(entity['end'] - 1)
        
        if start_token is not None and end_token is not None:
            label = entity['label']
            tags[start_token] = f"B-{label}"
            for i in range(start_token + 1, end_token + 1):
                if i < len(tags):
                    tags[i] = f"I-{label}"
    
    return tokens, tags

# Convert to token-level format
tokenized_examples = []
for example in examples:
    tokens, tags = create_ner_tags(example['text'], example['entities'])
    tokenized_examples.append({
        'tokens': tokens,
        'ner_tags': tags
    })

print(f"✅ Created {len(tokenized_examples)} tokenized examples")
print(f"\nSample:")
if tokenized_examples:
    sample = tokenized_examples[0]
    print(f"Tokens: {sample['tokens'][:10]}")
    print(f"Tags: {sample['ner_tags'][:10]}")

## 4️⃣ Create Dataset

In [ ]:
from datasets import Dataset, DatasetDict
import random

# Split into train/test (90/10)
random.shuffle(tokenized_examples)
split_idx = int(len(tokenized_examples) * 0.9)

train_data = tokenized_examples[:split_idx]
test_data = tokenized_examples[split_idx:]

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_dict({
    'tokens': [ex['tokens'] for ex in train_data],
    'ner_tags': [[label2id.get(tag, 0) for tag in ex['ner_tags']] for ex in train_data]
})

test_dataset = Dataset.from_dict({
    'tokens': [ex['tokens'] for ex in test_data],
    'ner_tags': [[label2id.get(tag, 0) for tag in ex['ner_tags']] for ex in test_data]
})

dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print(f"📊 Dataset created:")
print(f"  Train: {len(train_dataset)} examples")
print(f"  Test: {len(test_dataset)} examples")
print(f"\n{dataset}")

## 5️⃣ Load DeBERTa v3 Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

print(f"✅ Loaded {model_name}")
print(f"📊 Model size: {model.num_parameters() / 1e6:.1f}M parameters")

## 6️⃣ Tokenize Dataset

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )
    
    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset['train'].column_names
)

print("✅ Dataset tokenized")
print(tokenized_dataset)

## 7️⃣ Train Model

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForTokenClassification
import evaluate
import numpy as np

# Metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    predictions = np.argmax(predictions, axis=2)
    
    true_labels = [[LABELS[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [LABELS[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./resume-ner-deberta",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
    fp16=True,  # Use mixed precision on GPU
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🚀 Starting training...")
print(f"Total training steps: {len(tokenized_dataset['train']) * 5 // 8}")

In [ ]:
# Train the model
trainer.train()

print("\n✅ Training completed!")

## 8️⃣ Evaluate Model

In [ ]:
# Evaluate on test set
results = trainer.evaluate()

print("\n📊 Test Results:")
print(f"  Precision: {results['eval_precision']:.4f}")
print(f"  Recall: {results['eval_recall']:.4f}")
print(f"  F1: {results['eval_f1']:.4f}")
print(f"  Accuracy: {results['eval_accuracy']:.4f}")

## 9️⃣ Test Inference

In [ ]:
# Test on sample text
test_text = """Full Stack Developer Nov 2024 – Present
Lalataksha Consulting Services Pvt. Ltd. | Hyderabad, India
Built an AI-powered resume parsing system."""

# Tokenize
inputs = tokenizer(test_text, return_tensors="pt", truncation=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Predict
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=-1)[0].cpu().numpy()

# Decode
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
predicted_labels = [id2label[pred] for pred in predictions]

print("\n🔍 Sample Inference:")
print(f"Text: {test_text[:100]}...\n")
print("Token -> Predicted Label:")
for token, label in zip(tokens[:30], predicted_labels[:30]):
    if token not in ['[CLS]', '[SEP]', '[PAD]']:
        print(f"  {token:20s} -> {label}")

## 🔟 Save Model

In [ ]:
# Save model and tokenizer
output_dir = "./resume-ner-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

# Save label mappings
import json
with open(f"{output_dir}/label_mappings.json", "w") as f:
    json.dump({
        "labels": LABELS,
        "label_to_id": label2id,
        "id_to_label": id2label
    }, f, indent=2)

print(f"✅ Model saved to {output_dir}")
print(f"\nFiles saved:")
!ls -lh {output_dir}

## 📥 Download Model

In [ ]:
# Zip the model directory
!zip -r resume-ner-final.zip resume-ner-final/

# Download
from google.colab import files
files.download('resume-ner-final.zip')

print("\n✅ Model downloaded!")
print("\n📝 Next steps:")
print("1. Extract the zip file")
print("2. Copy to: ai-service/models/resume-ner-deberta/")
print("3. Restart your AI service")
print("4. Test with your resume!")